# XMF-GNN — Train, Evaluate, Ablate, Explain

Reproduces paper sections 4.4 (training), 4.6.1 (overall results),
4.6.3 (ablation), and 4.6.4 (Integrated Gradient feature importance + flow
attention trajectory).

Hyperparameters are taken from paper Table 3.

In [ ]:
import os
import torch
from torch_geometric.loader import DataLoader
from Utility import (
    NIDSDataset, XMFGNN, train, test, test_cm, calculate_metrics,
    DEFAULT_CIC_IOT2023_LABELS, collect_flow_attention,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

# Paper Table 3 hyperparameters.
args = {
    'device': device,
    'hidden_size': 64,
    'attn_size': 32,
    'epochs': 100,
    'lr': 1e-2,
    'min_lr': 1e-5,
    'scheduler_patience': 5,
    'scheduler_threshold': 0.01,
    'batch_size': 64,
}

In [ ]:
# Edit these paths to point at your processed dataset directories.
PROCESSED_ROOT = r'F:/CIC_IOT/processed_xmfgnn'
TRAIN_ROOT = os.path.join(PROCESSED_ROOT, 'train')
TEST_ROOT  = os.path.join(PROCESSED_ROOT, 'test')

train_set = NIDSDataset(root=TRAIN_ROOT, label_dict=DEFAULT_CIC_IOT2023_LABELS,
                        filename=['df_class_8_train.csv'], single_file=True,
                        skip_processing=True)
test_set  = NIDSDataset(root=TEST_ROOT, label_dict=DEFAULT_CIC_IOT2023_LABELS,
                        filename=['df_class_8_test.csv'], single_file=True,
                        skip_processing=True, test=True)

train_loader = DataLoader(train_set, batch_size=args['batch_size'], shuffle=True)
test_loader  = DataLoader(test_set,  batch_size=args['batch_size'])
print('train graphs:', len(train_set), 'test graphs:', len(test_set))

## 1. XMF-GNN (paper-faithful) — `fusion='attn'`

In [ ]:
sample = train_set[0]
model = XMFGNN(
    hetero_metadata=sample.metadata(),
    hidden_size=args['hidden_size'],
    attn_size=args['attn_size'],
    num_classes=len(DEFAULT_CIC_IOT2023_LABELS),
    fusion='attn',
).to(device)
print(model)

history = train(train_loader, model, args, device=device, val_loader=test_loader)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(history['train_acc'], label='Train Acc.')
ax[0].plot(history['val_acc'],   label='Test Acc.')
ax[0].set_xlabel('Epoch'); ax[0].set_ylabel('Accuracy'); ax[0].legend()
ax[0].set_title('(a) Accuracy vs Epoch')

ax[1].plot(history['loss'], color='red', label='Loss')
ax2 = ax[1].twinx()
ax2.plot(history['lr'], color='blue', linestyle='--', label='Learning Rate')
ax[1].set_xlabel('Epoch'); ax[1].set_ylabel('Loss'); ax2.set_ylabel('Learning Rate')
ax2.set_yscale('log')
ax[1].set_title('(b) Loss/LR vs Epoch')
fig.tight_layout(); plt.show()

In [ ]:
acc, preds, labels = test_cm(test_loader, model, device=device)
print('Final test accuracy:', acc)

## 2. Ablation study (paper Table 8)

In [ ]:
import time
import copy

ABLATION_VARIANTS = [
    ('B-GNN',   'baseline'),
    ('FM-GNN',  'mlp'),
    ('SA-GNN',  'simple_attn'),
    ('GA-GNN',  'gated'),
    ('MA-GNN',  'multi_head'),
    ('XMF-GNN', 'attn'),
]

ablation_results = {}
for tag, fusion in ABLATION_VARIANTS:
    print(f'\n=== {tag}  fusion={fusion} ===')
    m = XMFGNN(
        hetero_metadata=sample.metadata(),
        hidden_size=args['hidden_size'],
        attn_size=args['attn_size'],
        num_classes=len(DEFAULT_CIC_IOT2023_LABELS),
        fusion=fusion,
    ).to(device)
    t0 = time.time()
    h = train(train_loader, m, args, device=device, val_loader=test_loader)
    train_t = time.time() - t0
    t0 = time.time()
    acc = test(test_loader, m, device=device)
    test_t = time.time() - t0
    ablation_results[tag] = {
        'accuracy': acc,
        'training_seconds': train_t,
        'detecting_seconds': test_t,
    }

import pandas as pd
table = pd.DataFrame(ablation_results).T
table['overall_seconds'] = table['training_seconds'] + table['detecting_seconds']
print(table)

## 3. Flow-attention trajectory (paper Fig. 8)

Re-train each attention-equipped variant briefly while logging the average
`alpha_f` per epoch. Reproduces the paper's qualitative claim that the
dynamic-attention fusion converges to a stable mid-range alpha while the
multi-head variant degenerates to a quasi-static distribution.

In [ ]:
import numpy as np
from Utility import collect_flow_attention

FUSIONS_TO_TRACE = ['attn', 'simple_attn', 'gated', 'multi_head']
trace_args = dict(args); trace_args['epochs'] = 1

alpha_traces = {f: [] for f in FUSIONS_TO_TRACE}
for fusion in FUSIONS_TO_TRACE:
    print('Tracing', fusion)
    m = XMFGNN(
        hetero_metadata=sample.metadata(),
        hidden_size=args['hidden_size'], attn_size=args['attn_size'],
        num_classes=len(DEFAULT_CIC_IOT2023_LABELS), fusion=fusion,
    ).to(device)
    for epoch in range(args['epochs']):
        train(train_loader, m, trace_args, device=device, log_every=999)
        alpha_traces[fusion].append(collect_flow_attention(test_loader, m, device=device))

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (f, vals) in zip(axes.ravel(), alpha_traces.items()):
    ax.plot(vals); ax.set_title(f); ax.set_xlabel('epoch'); ax.set_ylabel('avg flow alpha')
fig.tight_layout(); plt.show()

## 4. Integrated Gradient feature importance (paper Fig. 6/7)

In [ ]:
from Utility.IG_Explainer import IntegratedGradientExplainer, top_flow_features, top_packet_features
import pandas as pd

# Reconstruct flow-feature names from the train CSV header.
import csv as _csv
TRAIN_CSV = os.path.join(TRAIN_ROOT, 'raw', train_set.raw_file_names[0])
with open(TRAIN_CSV, 'r', newline='') as f:
    header = next(_csv.reader(f))
flow_drop = {
    'udps.payload_data','udps.delta_time','udps.packet_direction',
    'udps.ip_size','udps.transport_size','udps.payload_size',
    'udps.syn','udps.cwr','udps.ece','udps.urg','udps.ack',
    'udps.psh','udps.rst','udps.fin','Label',
}
FLOW_FEATURE_NAMES = [c for c in header if c not in flow_drop]
print('flow feature dim:', len(FLOW_FEATURE_NAMES))

PACKET_PROTO_NAMES = [
    'direction','ip_size','transport_size','payload_size','delta_time',
    'syn','cwr','ece','urg','ack','psh','rst','fin','payload_density',
]

In [ ]:
ig = IntegratedGradientExplainer(model, device=device, n_steps=50)
attr = ig.explain(test_set[0])

print('predicted class:', attr.predicted_class, 'logprob:', attr.predicted_logprob)
print('\nTop flow features:')
for name, a, v in top_flow_features(attr, FLOW_FEATURE_NAMES, top_n=10):
    print(f'  {name:32s}  attr={a:+.4f}  value={v:+.4f}')
print('\nTop packet (protocol) features:')
for name, a, v in top_packet_features(attr, PACKET_PROTO_NAMES, top_n=10):
    print(f'  {name:24s}  attr={a:+.4f}  value={v:+.4f}')